# Plateau Failure Slice Audit

Stop-and-brainstorm gate after several pseudo-top regime-mean variants failed to produce a joint strict/stress improvement.

## 1. Audit Setup

This notebook compares completed artifacts rather than fitting a new model. It looks for repeated hard wells, progress-zone failures, analog-distance stress movement, and whether recent improvements are strict-only.

In [1]:

"Compare recent plateau runs by score, hard wells, progress zones, and analog-distance slices."

from pathlib import Path
import json
import shutil
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "context").exists():
    ROOT = next(parent for parent in ROOT.parents if (parent / "context").exists())

RUN_DATE = "2026-07-01"
RUN_TAG = "plateau_failure_slice_audit"
RUN_ID = f"{RUN_DATE}__{RUN_TAG}"
TABLE_DIR = ROOT / "artifacts" / "tables"
REPORT_DIR = ROOT / "artifacts" / "reports"
RUN_DIR = ROOT / "artifacts" / "logs" / "runs" / RUN_ID
for directory in [TABLE_DIR, REPORT_DIR, RUN_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

RUNS = {
    "regime_mean": "pseudo_top_regime_mean",
    "regime_tuned": "pseudo_top_regime_mean_tuning",
    "regime_stack": "xgboost_regime_mean_stack",
    "stress_gated_stack": "xgboost_regime_stress_gated_stack",
    "probability_scale": "pseudo_top_probability_scale_tuning",
    "more_rows": "pseudo_top_regime_mean_more_rows",
    "alignment_aware": "alignment_aware_pseudo_top_regime_mean",
    "partial_pooling": "bayesian_regime_partial_pooling",
}


def read_csv(name, suffix):
    path = TABLE_DIR / f"model__{name}_{suffix}__{RUN_DATE}.csv"
    if not path.exists():
        return pd.DataFrame()
    return pd.read_csv(path)

score_rows = []
well_frames = []
zone_frames = []
analog_frames = []
for short, name in RUNS.items():
    summary = read_csv(name, "summary")
    if summary.empty:
        continue
    best = summary.sort_values("rmse_mean").iloc[0]
    score_rows.append({
        "run_key": short,
        "artifact_name": name,
        "best_label": best.get("label", ""),
        "strict_rmse": float(best["rmse_mean"]),
        "stress_rmse": float(best.get("stress_rmse_mean", np.nan)),
        "stress_minus_strict": float(best.get("stress_minus_strict_rmse", np.nan)),
        "strict_delta_vs_regime_mean": float(best["rmse_mean"] - 13.092783707491265),
        "stress_delta_vs_regime_mean": float(best.get("stress_rmse_mean", np.nan) - 13.400574608144202),
    })
    wells = read_csv(name, "well_diagnostics")
    if not wells.empty:
        wells = wells.copy()
        wells["run_key"] = short
        wells["artifact_name"] = name
        wells["rank_within_run"] = wells["rmse"].rank(ascending=False, method="first")
        well_frames.append(wells)
    zones = read_csv(name, "zone_summary")
    if not zones.empty and "label" in zones.columns:
        zones = zones.loc[zones["label"].eq(best.get("label", ""))].copy()
        zones["run_key"] = short
        zones["artifact_name"] = name
        zone_frames.append(zones)
    analog = read_csv(name, "analog_bin_summary")
    if not analog.empty and "label" in analog.columns:
        analog = analog.loc[analog["label"].eq(best.get("label", ""))].copy()
        analog["run_key"] = short
        analog["artifact_name"] = name
        analog_frames.append(analog)

score_summary = pd.DataFrame(score_rows).sort_values("strict_rmse")
all_wells = pd.concat(well_frames, ignore_index=True) if well_frames else pd.DataFrame()
all_zones = pd.concat(zone_frames, ignore_index=True) if zone_frames else pd.DataFrame()
all_analog = pd.concat(analog_frames, ignore_index=True) if analog_frames else pd.DataFrame()

hard_well_consensus = pd.DataFrame()
if not all_wells.empty:
    top_wells = all_wells.loc[all_wells["rank_within_run"].le(15)].copy()
    hard_well_consensus = (
        top_wells.groupby("wellname")
        .agg(
            n_top15_runs=("run_key", "nunique"),
            mean_top_rmse=("rmse", "mean"),
            max_top_rmse=("rmse", "max"),
            runs=("run_key", lambda s: ", ".join(sorted(set(map(str, s))))),
        )
        .reset_index()
        .sort_values(["n_top15_runs", "mean_top_rmse"], ascending=[False, False])
    )

zone_comparison = pd.DataFrame()
if not all_zones.empty:
    keep = [c for c in ["run_key", "fold_column", "progress_zone", "n_rows", "rmse", "global_rmse", "regime_rmse", "baseline_rmse", "mean_blend_weight", "rmse_delta_vs_global"] if c in all_zones.columns]
    zone_comparison = all_zones[keep].sort_values(["fold_column", "progress_zone", "rmse"])

analog_comparison = pd.DataFrame()
if not all_analog.empty:
    keep = [c for c in ["run_key", "fold_column", "analog_distance_bin", "n_rows", "n_wells", "rmse", "global_rmse", "regime_rmse", "baseline_rmse", "mean_blend_weight", "rmse_delta_vs_global"] if c in all_analog.columns]
    analog_comparison = all_analog[keep].sort_values(["fold_column", "analog_distance_bin", "rmse"])

findings = pd.DataFrame([
    {
        "finding": "best_strict_is_fragile",
        "detail": "Residual stacks own the best strict score but worsen analog stress.",
        "value": 13.0769,
        "interpretation": "Do not chase strict-only residual gains without changing the state signal or stress objective.",
    },
    {
        "finding": "best_stress_still_original_regime_mean",
        "detail": "The original direct regime mean remains the cleanest strict/stress package-ready point.",
        "value": 13.4006,
        "interpretation": "Recent tuning, denser rows, alignment, and partial pooling did not improve the joint gate.",
    },
    {
        "finding": "next_model_direction",
        "detail": "Build a state-revision or local-log sequence model, not another shrinkage schedule.",
        "value": np.nan,
        "interpretation": "The plateau is more likely pseudo-top/state error than insufficient Ridge/Bayesian pooling.",
    },
])

paths = {
    "score_summary": TABLE_DIR / f"validation__{RUN_TAG}_score_summary__{RUN_DATE}.csv",
    "hard_well_consensus": TABLE_DIR / f"validation__{RUN_TAG}_hard_well_consensus__{RUN_DATE}.csv",
    "zone_comparison": TABLE_DIR / f"validation__{RUN_TAG}_zone_comparison__{RUN_DATE}.csv",
    "analog_comparison": TABLE_DIR / f"validation__{RUN_TAG}_analog_comparison__{RUN_DATE}.csv",
    "findings": TABLE_DIR / f"validation__{RUN_TAG}_findings__{RUN_DATE}.csv",
}
exports = {
    "score_summary": score_summary,
    "hard_well_consensus": hard_well_consensus,
    "zone_comparison": zone_comparison,
    "analog_comparison": analog_comparison,
    "findings": findings,
}
for key, frame in exports.items():
    frame.to_csv(paths[key], index=False)
    shutil.copy2(paths[key], RUN_DIR / paths[key].name)

def markdown_table(frame, columns):
    if frame.empty:
        return "No rows available."
    small = frame.loc[:, columns].copy()
    for col in small.select_dtypes(include=["float"]).columns:
        small[col] = small[col].map(lambda value: "" if pd.isna(value) else f"{value:.4f}")
    header = "| " + " | ".join(columns) + " |"
    sep = "| " + " | ".join(["---"] * len(columns)) + " |"
    rows = ["| " + " | ".join(str(row[col]) for col in columns) + " |" for _, row in small.iterrows()]
    return "\n".join([header, sep, *rows])


report_path = REPORT_DIR / f"validation__{RUN_TAG}__{RUN_DATE}.md"
score_md = markdown_table(score_summary, ["run_key", "strict_rmse", "stress_rmse", "strict_delta_vs_regime_mean", "stress_delta_vs_regime_mean"])
hard_md = markdown_table(hard_well_consensus.head(12), ["wellname", "n_top15_runs", "mean_top_rmse", "max_top_rmse", "runs"]) if not hard_well_consensus.empty else "No well diagnostics available."
report = f"""# Plateau Failure Slice Audit

## Score Pattern

{score_md}

## Hard-Well Consensus

{hard_md}

## Interpretation

Recent changes are not failing randomly. The direct pseudo-top regime mean remains the best joint strict/stress anchor, while strict-only residual stacks and alignment additions move error into stress-sensitive slices. The partial-pooling experiment is especially useful because it shows that sharing the current pseudo-top regime signal through one high-dimensional Ridge layer weakens the useful formation-specific mean.

## Next Plan

1. Stop simple shrinkage, row-density, and confidence-schedule sweeps.
2. Build a state-revision notebook that audits whether pseudo-top errors are systematic by well, progress zone, analog distance, GR motif, or typewell lag.
3. If systematic state errors exist, train a second-stage state corrector before TVT modeling.
4. If state errors are not systematic enough, try a local-log sequence model that predicts target deltas from known-zone and target-row windows directly.
"""
report_path.write_text(report, encoding="utf-8")
shutil.copy2(report_path, RUN_DIR / report_path.name)

print(score_summary.to_string(index=False))
print("\nHard wells")
print(hard_well_consensus.head(12).to_string(index=False))
print(report_path)


           run_key                          artifact_name                                                                   best_label  strict_rmse  stress_rmse  stress_minus_strict  strict_delta_vs_regime_mean  stress_delta_vs_regime_mean
stress_gated_stack      xgboost_regime_stress_gated_stack                                        blend_plus_ridge_residual_s15_uniform    13.076873    13.446788             0.369915                    -0.015911                     0.046214
      regime_stack              xgboost_regime_mean_stack                                                blend_plus_ridge_residual_s15    13.076873    13.446788             0.369915                    -0.015911                     0.046214
   alignment_aware alignment_aware_pseudo_top_regime_mean                     xgb_square_global__regime_ridge_alpha80__confidence_s100    13.083844    13.430270             0.346426                    -0.008940                     0.029696
      regime_tuned          pseudo_top_r

## 2. Result

The generated report and tables are stored under `artifacts/reports`, `artifacts/tables`, and the run bundle. The next modeling step should revise state inference or use a local-log sequence architecture rather than continuing shrinkage sweeps.